In [ ]:
# CELL 1: Install packages
# This notebook keeps the lab simple. We only need OpenAI SDK and python-dotenv.
# The full Lab 21 folder also includes tests, GitHub Actions, and deployment files.

%pip install -q openai==2.44.0 python-dotenv==1.2.2

In [ ]:
# CELL 2: Configure Azure OpenAI credentials
# Learners enter credentials at runtime so secrets are not stored in the notebook.
# In projects, use .env files, GitHub Actions secrets, Azure Key Vault, or another secret store.

import json
import os
import sys
from datetime import datetime
from getpass import getpass
from pathlib import Path

from openai import OpenAI


# Windows terminals sometimes cannot print special characters returned by the model.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


# Helper function: read a credential from environment first, then ask the learner.
# This keeps the notebook safe for GitHub because no key is saved in the file.
def read_secret(setting_name: str, prompt_text: str, hidden: bool = False) -> str:
    value = os.environ.get(setting_name, "").strip()
    if value:
        return value
    value = getpass(prompt_text).strip() if hidden else input(prompt_text).strip()
    if not value:
        raise ValueError(f"{setting_name} is required to run this notebook.")
    os.environ[setting_name] = value
    return value


os.environ["AZURE_OPENAI_ENDPOINT"] = read_secret(
    "AZURE_OPENAI_ENDPOINT",
    "Azure OpenAI endpoint, for example https://your-resource.openai.azure.com/openai/v1: ",
)
os.environ["AZURE_OPENAI_API_KEY"] = read_secret(
    "AZURE_OPENAI_API_KEY",
    "Azure OpenAI API key, hidden input: ",
    hidden=True,
)
os.environ["AZURE_OPENAI_API_VERSION"] = read_secret(
    "AZURE_OPENAI_API_VERSION",
    "Azure OpenAI API version, for example 2025-08-07: ",
)
os.environ["AZURE_OPENAI_DEPLOYMENT"] = read_secret(
    "AZURE_OPENAI_DEPLOYMENT",
    "Azure OpenAI deployment name, for example gpt-5-mini: ",
)

client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print("Azure OpenAI client is ready.")


In [ ]:
# CELL 3: Create three versioned prompt files
# Lab objective: Version and manage AI prompts using GitHub.
# This cell creates three prompt versions so learners can compare behavior.
# In GitHub, each prompt file and the manifest can be reviewed through commits and pull requests.

PROMPT_DIR = Path("notebook_prompts")
PROMPT_DIR.mkdir(exist_ok=True)

prompt_versions = {
    "1.0.0": {
        "prompt_file": "support_ops_prompt_v1.md",
        "description": "Basic support classification and next action.",
        "content": """
You are an enterprise support operations assistant.
Classify the support request and suggest the next operational action.
Do not ask for passwords, full card numbers, or one-time passcodes.
Keep the answer short and clear.
""".strip(),
    },
    "1.1.0": {
        "prompt_file": "support_ops_prompt_v2.md",
        "description": "Adds production incident severity and escalation guidance.",
        "content": """
You are an enterprise support operations assistant.
Classify the support request, assign severity, and suggest the next operational action.
Do not ask for passwords, full card numbers, or one-time passcodes.
If there is production impact, recommend escalation and identify the response team.
Include category, severity, and next action.
""".strip(),
    },
    "1.2.0": {
        "prompt_file": "support_ops_prompt_v3.md",
        "description": "Adds change-management and deployment-risk thinking.",
        "content": """
You are an enterprise support operations assistant working with DevOps teams.
Classify the support request, assign severity, and suggest the next operational action.
Do not ask for passwords, full card numbers, or one-time passcodes.
If there is production impact, recommend escalation, rollback investigation, and deployment/change review.
Include category, severity, next action, possible owner, and deployment-risk note.
""".strip(),
    },
}

manifest = {
    "prompt_name": "support_ops_prompt",
    "active_version": "1.1.0",
    "owner": "agentic-ops-team",
    "change_note": "Three prompt versions for Agentic Ops demo.",
    "versions": {
        version: {
            "prompt_file": details["prompt_file"],
            "description": details["description"],
        }
        for version, details in prompt_versions.items()
    },
}

for version, details in prompt_versions.items():
    (PROMPT_DIR / details["prompt_file"]).write_text(details["content"], encoding="utf-8")

(PROMPT_DIR / "prompt_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Prompt versions created:")
for version, details in manifest["versions"].items():
    print(f"- {version}: {details['prompt_file']} - {details['description']}")
print("Active version in manifest:", manifest["active_version"])

In [ ]:
# CELL 4: Build a simple AI agent using a selected prompt version
# Lab objective: Version and manage AI prompts using GitHub.
# Change SELECTED_PROMPT_VERSION to manually test different prompt versions.
# Try: 1.0.0, 1.1.0, or 1.2.0.

SELECTED_PROMPT_VERSION = "1.2.0"


# Function: load_versioned_prompt
# Purpose: Read the prompt manifest, select the requested prompt version,
# then load the matching prompt file from disk.
# Input: version such as "1.0.0", "1.1.0", or "1.2.0".
# Output: prompt text and prompt metadata for the selected version.
# Why it matters: This is the core prompt-versioning pattern used in Agentic Ops.
def load_versioned_prompt(version: str | None = None) -> tuple[str, dict]:
    """Load prompt text and metadata for the selected version."""
    manifest_data = json.loads((PROMPT_DIR / "prompt_manifest.json").read_text(encoding="utf-8"))
    selected_version = version or manifest_data["active_version"]

    if selected_version not in manifest_data["versions"]:
        available = sorted(manifest_data["versions"])
        raise ValueError(f"Unknown prompt version {selected_version}. Available versions: {available}")

    version_info = manifest_data["versions"][selected_version]
    prompt = (PROMPT_DIR / version_info["prompt_file"]).read_text(encoding="utf-8")
    version_info = {**version_info, "prompt_version": selected_version}
    return prompt, version_info


# Function: run_support_agent
# Purpose: Call Azure OpenAI using the selected versioned prompt.
# Input: user_request is the business/support problem; prompt_version chooses prompt behavior.
# Output: model response plus the prompt version details used for traceability.
# Why it matters: This shows how one agent can behave differently when prompt versions change.
def run_support_agent(user_request: str, prompt_version: str) -> str:
    """Call Azure OpenAI using the selected prompt version."""
    prompt, prompt_info = load_versioned_prompt(prompt_version)
    response = client.chat.completions.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_request},
        ],
    )
    return (
        f"Prompt version used: {prompt_info['prompt_version']}\n"
        f"Prompt file: {prompt_info['prompt_file']}\n"
        f"Description: {prompt_info['description']}\n\n"
        f"{response.choices[0].message.content or ''}"
    )


request = "Multiple customers report that checkout payment is failing and revenue is impacted."
print(run_support_agent(request, SELECTED_PROMPT_VERSION))

In [ ]:
# CELL 5: Simulate CI/CD checks for all prompt versions
# Lab objective: Create a CI/CD workflow for an AI application.
# In real GitHub Actions, these checks would run automatically on push or pull request.

# Function: check_prompt_manifest
# Purpose: Validate that prompt_manifest.json has the governance fields needed for review.
# Input: no direct input; it reads prompt_manifest.json from notebook_prompts/.
# Output: pass/fail boolean and explanation message.
# Why it matters: CI/CD should fail early if prompt metadata is incomplete.
def check_prompt_manifest() -> tuple[bool, str]:
    """Validate that the prompt manifest has required governance metadata."""
    manifest_data = json.loads((PROMPT_DIR / "prompt_manifest.json").read_text(encoding="utf-8"))
    required = {"prompt_name", "active_version", "owner", "change_note", "versions"}
    missing = sorted(required - set(manifest_data))
    if missing:
        return False, f"Missing manifest fields: {missing}"
    if manifest_data["active_version"] not in manifest_data["versions"]:
        return False, "Active version is not listed in versions."
    return True, "Prompt manifest has required metadata and active version."


# Function: check_prompt_versions
# Purpose: Validate that every prompt version listed in the manifest has a real file.
# It also checks that every prompt has basic safety and action guidance.
# Input: no direct input; it reads all prompt versions from the manifest.
# Output: pass/fail boolean and explanation message.
# Why it matters: This simulates automated prompt governance in a CI/CD pipeline.
def check_prompt_versions() -> tuple[bool, str]:
    """Validate that every prompt version listed in the manifest has a file and useful guidance."""
    manifest_data = json.loads((PROMPT_DIR / "prompt_manifest.json").read_text(encoding="utf-8"))
    required_terms = ["passwords", "support", "action"]

    for version, details in manifest_data["versions"].items():
        prompt_file = PROMPT_DIR / details["prompt_file"]
        if not prompt_file.exists():
            return False, f"Prompt file missing for version {version}: {prompt_file}"
        prompt = prompt_file.read_text(encoding="utf-8").lower()
        missing_terms = [term for term in required_terms if term not in prompt]
        if missing_terms:
            return False, f"Version {version} missing terms: {missing_terms}"

    return True, "All prompt versions exist and contain required guidance."


# Function: run_ci_cd_checks
# Purpose: Run all validation checks before allowing deployment.
# Input: no direct input; it calls the individual check functions.
# Output: list of check names, pass/fail results, and messages.
# Why it matters: This represents the automated quality gate in GitHub Actions.
def run_ci_cd_checks() -> list[tuple[str, bool, str]]:
    """Run simple CI/CD checks before deployment."""
    checks = []
    for name, check_fn in [
        ("prompt_manifest_validation", check_prompt_manifest),
        ("prompt_versions_validation", check_prompt_versions),
    ]:
        passed, message = check_fn()
        checks.append((name, passed, message))
    return checks


ci_results = run_ci_cd_checks()
for name, passed, message in ci_results:
    print(f"{name}: {'PASS' if passed else 'FAIL'} - {message}")

In [ ]:
# CELL 6: Simulate deployment pipeline
# Lab objective: Build a simple automated deployment pipeline for AI agents.
# Deployment is allowed only when every CI/CD check passes.
# The deployment record captures which prompt version was deployed.

# Function: deploy_agent
# Purpose: Simulate an automated deployment decision for the AI agent.
# Input: CI/CD check results and selected prompt version.
# Output: deployment record showing environment, agent name, prompt version, and status.
# Why it matters: Deployment should be allowed only when validation checks pass.
def deploy_agent(checks: list[tuple[str, bool, str]], prompt_version: str) -> dict:
    """Create a simple deployment record after checking all validations."""
    all_passed = all(passed for _, passed, _ in checks)
    return {
        "deployment_time": datetime.now().isoformat(timespec="seconds"),
        "environment": "staging",
        "agent_name": "support-ops-agent",
        "prompt_version": prompt_version,
        "status": "DEPLOYED_TO_STAGING" if all_passed else "DEPLOYMENT_BLOCKED",
    }


deployment_record = deploy_agent(ci_results, SELECTED_PROMPT_VERSION)
print(json.dumps(deployment_record, indent=2))

In [ ]:
# CELL 7: GitHub Actions workflow example
# This cell does not call GitHub. It only shows what a minimal workflow file could look like.
# Learners can copy this into .github/workflows/ai-agent-ci-cd.yml in a real repository.

github_actions_yaml = """
name: AI Agent CI CD

on:
  push:
    branches: [main]
  pull_request:

jobs:
  validate-agent:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest
""".strip()

print(github_actions_yaml)

In [ ]:
# CELL 8: Optional GitHub credentials for live deployment demo
# This cell lets learners connect the notebook to their own GitHub repository.
# The access token is entered with getpass, so it is not displayed in notebook output.
# Required token scopes for this demo: repo and workflow.
# Do not hard-code the GitHub token in a notebook that will be shared.

import base64
import shutil
import subprocess
from getpass import getpass


GIT_USER_NAME = input("Git commit user name, for example Abhishek Sri: " ).strip()
GITHUB_REPO_URL = input("GitHub repo URL, for example https://github.com/user/repo.git: " ).strip()
GIT_USER_EMAIL = input("Git commit email: " ).strip()
GITHUB_TOKEN = getpass("GitHub access token, hidden input: " )

if not GIT_USER_NAME or not GITHUB_REPO_URL or not GIT_USER_EMAIL or not GITHUB_TOKEN:
    raise ValueError("Git commit name, repo URL, email, and token are required for the live demo.")

print("GitHub credentials captured securely for this notebook session.")
print("Token is stored only in memory and is not printed.")

In [ ]:
# CELL 9: Push prompt, CI workflow, and deployment manifest to GitHub
# This cell demonstrates a real DevOps flow:
# 1. Clone the GitHub repository.
# 2. Create a feature branch.
# 3. Add versioned prompt files.
# 4. Add a GitHub Actions workflow.
# 5. Add a deployment manifest.
# 6. Commit and push the branch.
#
# After this cell succeeds, open GitHub and check the branch and Actions tab.

LIVE_DEMO_DIR = Path("github_live_deployment_demo")
BRANCH_NAME = "feature/notebook-agentic-ops-demo"


# Function: run_command
# Purpose: Run Git commands from the notebook and stop with a clear error if a command fails.
# Input: command list, optional working directory, and whether GitHub token auth is needed.
# Output: subprocess result object containing stdout, stderr, and return code.
# Why it matters: This makes the live GitHub demo repeatable and safer for learners.
def run_command(command: list[str], cwd: Path | None = None, use_token: bool = False) -> subprocess.CompletedProcess:
    """Run a command and raise a learner-friendly error if it fails."""
    safe_command = command[:]

    if use_token:
        token_text = f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
        auth_header = base64.b64encode(token_text).decode("utf-8")
        command = [
            "git",
            "-c",
            f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth_header}",
            *command[1:],
        ]
        safe_command = ["git", "-c", "http.https://github.com/.extraheader=AUTHORIZATION: basic ***", *safe_command[1:]]

    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print("Command failed:", " ".join(safe_command))
        print(result.stdout)
        print(result.stderr.replace(GITHUB_TOKEN, "***"))
        raise RuntimeError("GitHub live deployment command failed.")
    return result


if LIVE_DEMO_DIR.exists():
    shutil.rmtree(LIVE_DEMO_DIR)

run_command(["git", "clone", GITHUB_REPO_URL, str(LIVE_DEMO_DIR)], use_token=True)
run_command(["git", "config", "user.name", GIT_USER_NAME], cwd=LIVE_DEMO_DIR)
run_command(["git", "config", "user.email", GIT_USER_EMAIL], cwd=LIVE_DEMO_DIR)
run_command(["git", "checkout", "-B", BRANCH_NAME], cwd=LIVE_DEMO_DIR)

(LIVE_DEMO_DIR / "prompts").mkdir(parents=True, exist_ok=True)
(LIVE_DEMO_DIR / "deployment").mkdir(parents=True, exist_ok=True)
(LIVE_DEMO_DIR / ".github" / "workflows").mkdir(parents=True, exist_ok=True)

for prompt_file in PROMPT_DIR.glob("support_ops_prompt_*.md"):
    shutil.copy2(prompt_file, LIVE_DEMO_DIR / "prompts" / prompt_file.name)
shutil.copy2(PROMPT_DIR / "prompt_manifest.json", LIVE_DEMO_DIR / "prompts" / "prompt_manifest.json")

deployment_manifest = {
    "application": "support-ops-agent",
    "environment": "staging",
    "prompt_version": SELECTED_PROMPT_VERSION,
    "deployment_type": "notebook-live-demo",
    "created_by": GIT_USER_NAME,
}
(LIVE_DEMO_DIR / "deployment" / "deployment_manifest.json").write_text(
    json.dumps(deployment_manifest, indent=2),
    encoding="utf-8",
)

workflow_text = """name: Notebook Agentic Ops CI

on:
  push:
    branches:
      - main
      - feature/**
  pull_request:

jobs:
  validate-prompt-and-deployment:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Validate prompt and deployment files
        run: |
          python - <<'PY'
          import json
          from pathlib import Path

          manifest = json.loads(Path('prompts/prompt_manifest.json').read_text())
          required = {'prompt_name', 'active_version', 'owner', 'change_note', 'versions'}
          missing = required - set(manifest)
          assert not missing, f'Missing manifest fields: {missing}'
          for version, details in manifest['versions'].items():
              assert Path('prompts', details['prompt_file']).exists(), f'Prompt file missing for {version}'
          deployment = json.loads(Path('deployment/deployment_manifest.json').read_text())
          assert deployment['environment'] == 'staging'
          print('Prompt and deployment validation passed.')
          PY
"""
(LIVE_DEMO_DIR / ".github" / "workflows" / "notebook-agentic-ops-ci.yml").write_text(workflow_text, encoding="utf-8")

readme_text = f"""# Notebook Agentic Ops Deployment Demo

This repository branch was created from the Lab 21 beginner notebook.

## What was deployed

- Versioned prompts: `prompts/support_ops_prompt_v1.md`, `support_ops_prompt_v2.md`, `support_ops_prompt_v3.md`
- Prompt manifest: `prompts/prompt_manifest.json`
- Deployment manifest: `deployment/deployment_manifest.json`
- GitHub Actions workflow: `.github/workflows/notebook-agentic-ops-ci.yml`

Selected prompt version: `{SELECTED_PROMPT_VERSION}`
Target environment: `staging`
"""
(LIVE_DEMO_DIR / "README.md").write_text(readme_text, encoding="utf-8")

run_command(["git", "add", "."], cwd=LIVE_DEMO_DIR)
status = run_command(["git", "status", "--short"], cwd=LIVE_DEMO_DIR).stdout.strip()
print("Files prepared for commit:")
print(status or "No changes detected.")

if status:
    run_command(["git", "commit", "-m", "Add notebook Agentic Ops deployment demo"], cwd=LIVE_DEMO_DIR)
    run_command(["git", "push", "-u", "origin", BRANCH_NAME, "--force-with-lease"], cwd=LIVE_DEMO_DIR, use_token=True)

repo_web_url = GITHUB_REPO_URL.replace(".git", "")
print("Live deployment demo branch pushed successfully.")
print("Branch URL:", f"{repo_web_url}/tree/{BRANCH_NAME}")
print("Actions URL:", f"{repo_web_url}/actions")

In [ ]:
# CELL 10: Check GitHub Actions status from the notebook
# This uses the GitHub REST API instead of GitHub CLI, so it works in most notebook environments.
# Wait 20-40 seconds after pushing before running this cell.

from urllib.request import Request, urlopen


# Function: parse_repo_owner_name
# Purpose: Convert a GitHub repository URL into owner and repository name.
# Input: GitHub URL such as https://github.com/user/repo.git.
# Output: owner and repo strings used by the GitHub REST API.
# Why it matters: The Actions status API needs owner/repo format, not the full clone URL.
def parse_repo_owner_name(repo_url: str) -> tuple[str, str]:
    """Extract owner and repository name from a GitHub HTTPS URL."""
    cleaned = repo_url.replace("https://github.com/", "").replace(".git", "").strip("/")
    owner, repo = cleaned.split("/", 1)
    return owner, repo


owner, repo = parse_repo_owner_name(GITHUB_REPO_URL)
api_url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs?branch={BRANCH_NAME}&per_page=3"
request = Request(
    api_url,
    headers={
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    },
)

with urlopen(request, timeout=20) as response:
    data = json.loads(response.read().decode("utf-8"))

runs = data.get("workflow_runs", [])
if not runs:
    print("No workflow run found yet. Wait a few seconds and run this cell again.")
else:
    for run in runs:
        print("Workflow:", run.get("name"))
        print("Status:", run.get("status"))
        print("Conclusion:", run.get("conclusion"))
        print("URL:", run.get("html_url"))
        print("---")